# ***Exploratory Data Analysis***

In [1]:
import os
from dotenv import load_dotenv
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Load environment variables from .env file
load_dotenv()

# Set the path to the JDBC driver JAR file
JDBC_JAR_PATH = r"C:\Users\91852\OneDrive\Desktop\Museum\drivers\postgresql.jar"

# Create a SparkSession
spark = (
    SparkSession.builder
    .appName("MuseumEDA")
    .master("local[*]")
    .config("spark.driver.extraClassPath", JDBC_JAR_PATH)
    .config("spark.executor.extraClassPath", JDBC_JAR_PATH)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

JDBC_URL = f"jdbc:postgresql://{os.getenv('POSTGRES_HOST', 'localhost')}:{os.getenv('POSTGRES_PORT', '5432')}/{os.getenv('POSTGRES_DATABASE', 'Museum')}"
DB_USER = os.getenv('POSTGRES_USERNAME')
DB_PASS = os.getenv('POSTGRES_PASSWORD')

# Helper function to read a table from PostgreSQL into a Spark DataFrame
def load_table(table_name):
    return (
        spark.read
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", f'"bronze"."{table_name}"')
        .option("user", DB_USER)
        .option("password", DB_PASS)
        .option("driver", "org.postgresql.Driver")
        .load()
    )

print("PySpark initialized and ready to query Postgres!")

PySpark initialized and ready to query Postgres!


In [4]:
# Public tables check
schema_query = """
(
    SELECT
        table_name,
        COUNT(column_name) AS number_of_columns
    FROM information_schema.columns
    WHERE table_schema = 'public'
    GROUP BY table_name
    ORDER BY table_name
) AS schema_metadata
"""

schema_df = (
    spark.read
    .format("jdbc")
    .option("url", JDBC_URL)
    .option("dbtable", schema_query)
    .option("user", DB_USER)
    .option("password", DB_PASS)
    .option("driver", "org.postgresql.Driver")
    .load()
).toPandas()

print("Calculating exact row counts...")

row_counts = []

for table in schema_df["table_name"]:
    count = (
        spark.read
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", f'"public"."{table}"')
        .option("user", DB_USER)
        .option("password", DB_PASS)
        .option("driver", "org.postgresql.Driver")
        .load()
        .count()
    )

    row_counts.append(count)

schema_df["number_of_rows"] = row_counts

schema_df = (
    schema_df
    .sort_values("number_of_rows", ascending=False)
    .reset_index(drop=True)
)

schema_df["number_of_rows"] = schema_df["number_of_rows"].map("{:,}".format)

display(schema_df)

Calculating exact row counts...


,table_name,number_of_columns,number_of_rows
0,order_items,8,"4,722"
1,orders,9,"1,615"
2,customers,10,"1,445"
3,stocks,4,939
4,products,7,321
5,brands,3,10
6,staffs,9,10
7,categories,3,7
8,stores,9,3


In [6]:
from pyspark.sql.functions import col, sum as spark_sum, when

# Get all public tables
tables_query = """
(
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
      AND table_type = 'BASE TABLE'
) AS public_tables
"""

tables_df = (
    spark.read
    .format("jdbc")
    .option("url", JDBC_URL)
    .option("dbtable", tables_query)
    .option("user", DB_USER)
    .option("password", DB_PASS)
    .option("driver", "org.postgresql.Driver")
    .load()
)

results = []

for row in tables_df.collect():
    table_name = row["table_name"]

    print(f"Checking {table_name}...")

    df = (
        spark.read
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", f'"public"."{table_name}"')
        .option("user", DB_USER)
        .option("password", DB_PASS)
        .option("driver", "org.postgresql.Driver")
        .load()
    )

    total_rows = df.count()

    for field in df.schema.fields:
        column_name = field.name
        data_type = str(field.dataType)

        null_count = (
            df.select(
                spark_sum(
                    when(col(column_name).isNull(), 1).otherwise(0)
                ).alias("nulls")
            )
            .collect()[0]["nulls"]
        )

        results.append({
            "table_name": table_name,
            "column_name": column_name,
            "data_type": data_type,
            "total_rows": total_rows,
            "null_count": null_count,
            "null_pct": round((null_count / total_rows) * 100, 2) if total_rows > 0 else 0
        })

quality_df = pd.DataFrame(results)

display(quality_df.sort_values(
    ["table_name", "null_count"],
    ascending=[True, False]
))

Checking stores...
Checking stocks...
Checking orders...
Checking order_items...
Checking products...
Checking staffs...
Checking brands...
Checking categories...
Checking customers...


,table_name,column_name,data_type,total_rows,null_count,null_pct
46,brands,brand_id,LongType(),10,0,0.0
47,brands,brand_name,StringType(),10,0,0.0
48,brands,updated_at,TimestampType(),10,0,0.0
49,categories,category_id,LongType(),7,0,0.0
50,categories,category_name,StringType(),7,0,0.0
...,...,...,...,...,...,...
4,stores,street,StringType(),3,0,0.0
5,stores,city,StringType(),3,0,0.0
6,stores,state,StringType(),3,0,0.0
7,stores,zip_code,LongType(),3,0,0.0
